# ORBITGUARD Stage 3: Data preprocessing and feature engineering

This notebook prepares the OPSSAT-AD segment table for later classification. It uses the official `train` split, keeps the original `anomaly` labels, and writes processed train/test files under `data/processed/`. It does **not** train a model, resample classes, remove outliers, scale features, or modify the original downloaded data.

Run the cells from top to bottom in VS Code or Google Colab. The Stage 1 file `data/raw/dataset.csv` must be available in the project.

## 1. Set up paths and load the verified dataset

The notebook finds the project folder from the current location and imports the reusable preprocessing functions. In Colab, first open or clone the ORBITGUARD project so its `src` and `data/raw` folders are available. The original Stage 1 CSV remains read-only in this workflow.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

current_path = Path.cwd().resolve()
if current_path.name.lower() == "notebooks":
    current_path = current_path.parent

PROJECT_ROOT = next(
    (candidate for candidate in (current_path, *current_path.parents)
     if (candidate / "src" / "preprocessing.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the ORBITGUARD project (src/preprocessing.py). "
        "Open this notebook from the project or clone the repository in Colab."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import (
    ENGINEERED_FEATURE_COLUMNS,
    SPLIT_COLUMN,
    TARGET_COLUMN,
    load_feature_dataset,
)
from src.preprocessing import (
    EXCLUDED_COLUMNS,
    inspect_dataset,
    prepare_dataset,
)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

source_data = load_feature_dataset(RAW_DATA_DIR)
print(f"Project folder: {PROJECT_ROOT}")
print(f"Original feature dataset shape: {source_data.shape}")
display(source_data.head())

Project folder: C:\Users\HP\OneDrive\Desktop\orbit gaurd
Original feature dataset shape: (2123, 23)


,segment,anomaly,train,channel,sampling,duration,len,mean,var,std,...,smooth10_n_peaks,smooth20_n_peaks,diff_peaks,diff2_peaks,diff_var,diff2_var,gaps_squared,len_weighted,var_div_duration,var_div_len
0,1,1,1,CADC0872,1,279,280,8.533143e-07,3.494283e-10,0.000019,...,3,2,4,6,1.271176e-10,2.960666e-10,309,280,1.252431e-12,1.247958e-12
1,2,1,1,CADC0872,1,476,477,-3.639396e-06,6.476485e-10,0.000025,...,1,1,5,8,1.489383e-12,3.004752e-12,644,477,1.360606e-12,1.357754e-12
2,3,1,1,CADC0872,1,594,595,1.170788e-05,5.592877e-10,0.000024,...,2,2,2,3,4.112280e-12,1.029918e-11,772,595,9.415618e-13,9.399794e-13
3,4,1,1,CADC0872,1,271,272,8.486808e-07,5.466024e-10,0.000023,...,2,2,3,6,2.475760e-11,6.240985e-11,339,272,2.016983e-12,2.009568e-12
4,5,0,0,CADC0872,1,255,257,1.058485e-05,5.279023e-10,0.000023,...,1,1,78,87,5.547101e-13,7.035422e-13,357,257,2.070205e-12,2.054094e-12


## 2. Identify prediction fields, metadata, and the official split

`anomaly` is the answer the future classifier will learn to predict. `train` identifies the benchmark's official train/test partition; it is not a prediction input. `segment` identifies a segment, while `channel` and `sampling` describe its source/acquisition. These identifiers and metadata are excluded from model inputs. We use the documented 18 numerical benchmark features and preserve the original target values.

In [2]:
print("Actual columns:", source_data.columns.tolist())
print("\nColumn data types:")
display(source_data.dtypes.astype(str).rename("dtype").to_frame())

present_excluded = [column for column in EXCLUDED_COLUMNS if column in source_data.columns]
print("Identifier and metadata columns excluded:", ["segment", "channel", "sampling"])
print(f"Target column (kept separately): {TARGET_COLUMN}")
print(f"Official split column (used only to partition rows): {SPLIT_COLUMN}")
print(f"Excluded from the feature matrix: {present_excluded}")
print(f"Documented numeric benchmark feature count: {len(ENGINEERED_FEATURE_COLUMNS)}")
print("Feature names:", list(ENGINEERED_FEATURE_COLUMNS))

split_labels = source_data[SPLIT_COLUMN].map({1: "Training", 0: "Testing"})
print("\nOfficial split counts:")
display(split_labels.value_counts().rename_axis("split").to_frame("segments"))
print("Target counts within the official split:")
display(pd.crosstab(split_labels, source_data[TARGET_COLUMN]).rename_axis("split"))

Actual columns: ['segment', 'anomaly', 'train', 'channel', 'sampling', 'duration', 'len', 'mean', 'var', 'std', 'kurtosis', 'skew', 'n_peaks', 'smooth10_n_peaks', 'smooth20_n_peaks', 'diff_peaks', 'diff2_peaks', 'diff_var', 'diff2_var', 'gaps_squared', 'len_weighted', 'var_div_duration', 'var_div_len']

Column data types:


,dtype
segment,int64
anomaly,int64
train,int64
channel,str
sampling,int64
duration,int64
len,int64
mean,float64
var,float64
std,float64


Identifier and metadata columns excluded: ['segment', 'channel', 'sampling']
Target column (kept separately): anomaly
Official split column (used only to partition rows): train
Excluded from the feature matrix: ['segment', 'channel', 'sampling', 'anomaly', 'train']
Documented numeric benchmark feature count: 18
Feature names: ['duration', 'len', 'mean', 'var', 'std', 'kurtosis', 'skew', 'n_peaks', 'smooth10_n_peaks', 'smooth20_n_peaks', 'diff_peaks', 'diff2_peaks', 'diff_var', 'diff2_var', 'gaps_squared', 'len_weighted', 'var_div_duration', 'var_div_len']

Official split counts:


,segments
split,
Training,1594
Testing,529


Target counts within the official split:


anomaly,0,1
split,,
Testing,416,113
Training,1273,321


## 3. Check missing values and duplicate records

We count missing values by source column and look for exact duplicate rows, duplicate feature vectors, and repeated segment IDs. Duplicate detection is diagnostic: records are not silently dropped, because doing so could alter the benchmark or its labels. If rows need investigation, it should be done explicitly.

In [3]:
quality = inspect_dataset(source_data)
print(f"Dataset shape: {quality['shape'][0]:,} rows × {quality['shape'][1]} columns")
print("Missing values by column:")
display(quality["missing_by_column"].rename("missing_count").to_frame())
print(f"Exact duplicate rows: {quality['duplicate_rows']}")
print(f"Duplicate rows across the 18 benchmark feature values: {quality['duplicate_feature_rows']}")
print(f"Duplicate segment identifiers: {quality['duplicate_segment_ids']}")

Dataset shape: 2,123 rows × 23 columns
Missing values by column:


,missing_count
segment,0
anomaly,0
train,0
channel,0
sampling,0
duration,0
len,0
mean,0
var,0
std,0


Exact duplicate rows: 0
Duplicate rows across the 18 benchmark feature values: 0
Duplicate segment identifiers: 0


## 4. Investigate outliers without discarding useful telemetry

Stage 2 used the 1.5×IQR rule to flag unusually distant values. A large value can be a genuine telemetry event or a useful anomaly signal, so this workflow reports IQR outlier counts on the **training partition only** and retains every value. Computing the diagnostic on training only also avoids using test-distribution information in preprocessing decisions.

## 5. Prepare features with training-only median imputation

The downloaded feature table currently has no missing values. We nevertheless set up median imputation as a reproducible safeguard for missing or non-finite numeric values if they occur in a later compatible dataset. The imputer is fitted only on training rows and then applied to test rows. The target and split are never imputed. No scaling is applied: it is not required for common tree-based classifiers, and scaling can be chosen later if a model needs it. No records are oversampled or undersampled.

In [4]:
prepared = prepare_dataset(source_data, output_dir=PROCESSED_DATA_DIR)

print("Potential outlier counts in training rows (retained, not removed):")
display(prepared.outlier_summary)

print("Training imputation medians learned from training features only:")
imputation_medians = pd.Series(
    prepared.imputer.statistics_, index=ENGINEERED_FEATURE_COLUMNS, name="training_median"
)
display(imputation_medians.to_frame())

print("Processed files saved:")
for filename in ["train_processed.csv", "test_processed.csv", "preprocessing_metadata.json", "preprocessing_pipeline.joblib"]:
    print(f"- {PROCESSED_DATA_DIR / filename}")

Potential outlier counts in training rows (retained, not removed):


,potential_outlier_count,percent_of_training_rows
n_peaks,346,21.71
diff_var,331,20.77
smooth20_n_peaks,274,17.19
var,259,16.25
diff2_var,243,15.24
var_div_len,242,15.18
diff_peaks,197,12.36
smooth10_n_peaks,137,8.59
var_div_duration,132,8.28
diff2_peaks,122,7.65


Training imputation medians learned from training features only:


,training_median
duration,2.250000e+02
len,6.850000e+01
mean,4.488836e-06
var,6.174950e-10
std,2.484945e-05
kurtosis,-1.299938e+00
skew,9.950425e-03
n_peaks,1.000000e+00
smooth10_n_peaks,1.000000e+00
smooth20_n_peaks,1.000000e+00


Processed files saved:
- C:\Users\HP\OneDrive\Desktop\orbit gaurd\data\processed\train_processed.csv
- C:\Users\HP\OneDrive\Desktop\orbit gaurd\data\processed\test_processed.csv
- C:\Users\HP\OneDrive\Desktop\orbit gaurd\data\processed\preprocessing_metadata.json
- C:\Users\HP\OneDrive\Desktop\orbit gaurd\data\processed\preprocessing_pipeline.joblib


## 6. Check for target and split leakage

The feature matrix is built from an explicit allowlist of the 18 benchmark measurements. We verify that the target, split, identifier, and metadata columns are absent from both feature matrices, and that training and test rows do not overlap. The `anomaly` labels remain in separate target series and in the saved processed tables.

In [5]:
feature_names = list(prepared.X_train.columns)
leaked_columns = sorted(set(feature_names).intersection(EXCLUDED_COLUMNS))
assert not leaked_columns, f"Excluded columns leaked into model inputs: {leaked_columns}"
assert feature_names == list(ENGINEERED_FEATURE_COLUMNS)
assert prepared.leakage_check_passed
assert source_data.loc[prepared.y_train.index, TARGET_COLUMN].equals(prepared.y_train)
assert source_data.loc[prepared.y_test.index, TARGET_COLUMN].equals(prepared.y_test)
assert not prepared.X_train.index.intersection(prepared.X_test.index).size
assert np.isfinite(prepared.X_train.to_numpy()).all()
assert np.isfinite(prepared.X_test.to_numpy()).all()
assert not prepared.X_train.isna().any().any()
assert not prepared.X_test.isna().any().any()

print("Target leakage check: PASSED")
print("- anomaly is absent from X_train and X_test; labels are separate and unchanged.")
print("- train is absent from X_train and X_test; only used to preserve the official split.")
print("- segment, channel, and sampling are absent from model features.")
print("- training and test row indexes do not overlap.")
print("- imputer was fitted on training features and only transformed test features.")

Target leakage check: PASSED
- anomaly is absent from X_train and X_test; labels are separate and unchanged.
- train is absent from X_train and X_test; only used to preserve the official split.
- segment, channel, and sampling are absent from model features.
- training and test row indexes do not overlap.
- imputer was fitted on training features and only transformed test features.


## 7. Final prepared dataset report

These are the feature and target dimensions after preprocessing. The class counts should match the official benchmark split, and missing values should be absent from the model input matrices. This is the end of Stage 3; no classifier is trained.

In [6]:
def target_count_table(target):
    """Show both expected classes, including a zero count if one is absent."""
    return target.value_counts().reindex([0, 1], fill_value=0).rename_axis("anomaly").to_frame("count")

print("Final feature names:")
print(feature_names)
print(f"\nTraining feature shape: {prepared.X_train.shape}")
print(f"Testing feature shape:  {prepared.X_test.shape}")
print(f"Training target shape:  {prepared.y_train.shape}")
print(f"Testing target shape:   {prepared.y_test.shape}")
print("\nTraining class counts:")
display(target_count_table(prepared.y_train))
print("Testing class counts:")
display(target_count_table(prepared.y_test))
print(f"Missing feature values after preprocessing — train: {int(prepared.X_train.isna().sum().sum())}; "
      f"test: {int(prepared.X_test.isna().sum().sum())}")
print(f"Target leakage checked: {'YES' if prepared.leakage_check_passed else 'NO'}")

saved_train = pd.read_csv(PROCESSED_DATA_DIR / "train_processed.csv")
saved_test = pd.read_csv(PROCESSED_DATA_DIR / "test_processed.csv")
assert saved_train.shape == (len(prepared.X_train), len(feature_names) + 1)
assert saved_test.shape == (len(prepared.X_test), len(feature_names) + 1)
assert saved_train[TARGET_COLUMN].value_counts().sort_index().to_dict() == prepared.y_train.value_counts().sort_index().to_dict()
assert saved_test[TARGET_COLUMN].value_counts().sort_index().to_dict() == prepared.y_test.value_counts().sort_index().to_dict()
print("\nSaved train/test CSVs reloaded and verified. Stage 3 complete; no model was trained.")

Final feature names:
['duration', 'len', 'mean', 'var', 'std', 'kurtosis', 'skew', 'n_peaks', 'smooth10_n_peaks', 'smooth20_n_peaks', 'diff_peaks', 'diff2_peaks', 'diff_var', 'diff2_var', 'gaps_squared', 'len_weighted', 'var_div_duration', 'var_div_len']

Training feature shape: (1594, 18)
Testing feature shape:  (529, 18)
Training target shape:  (1594,)
Testing target shape:   (529,)

Training class counts:


,count
anomaly,
0,1273
1,321


Testing class counts:


,count
anomaly,
0,416
1,113


Missing feature values after preprocessing — train: 0; test: 0
Target leakage checked: YES

Saved train/test CSVs reloaded and verified. Stage 3 complete; no model was trained.
